In [ ]:
!pip install scikit-learn matplotlib seaborn xgboost

### Data Loading & Split (Training & Testing)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load the feature-engineered dataset we created in step 04
df = pd.read_csv('../data/processed/cleaned_data.csv')

# 2. Select leakage-free features (Log_Total_Funding is allowed as a predictor feature)
features_classifier = [
    'Total_Funding_Rounds', 'Unique_Investors_Count', 'Fought_Through_Recession',
    'Age_at_Latest_Round', 'Log_Total_Funding', 'Max_Investor_PageRank', 'Sum_Investor_PageRank',
    'country_code_cleaned', 'Industry_Sector_cleaned'
]

X = df[features_classifier].copy()
y = df['is_successful']

# One-hot encode categoricals
X = pd.get_dummies(X, columns=['country_code_cleaned', 'Industry_Sector_cleaned'], drop_first=False)

# 3. Train-Test Split (80% training data, 20% testing data, stratified due to imbalance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Data successfully split for training!")
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

### Model Training (Random Forest Classifier)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Calculate class weight ratio
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
scale_weight = num_neg / num_pos

# 1. Initialize XGBoost Classifier
model = XGBClassifier(
    n_estimators=150, 
    max_depth=5, 
    learning_rate=0.08,
    scale_pos_weight=scale_weight,
    random_state=42, 
    eval_metric='logloss'
)

# 2. Train the model
print("Training the XGBoost model... Please wait a moment.")
model.fit(X_train, y_train)

# 3. Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\nModel Training Completed!")
print(f"Accuracy Score: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### Feature Importance & Saving the Model

In [ ]:
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Plot Feature Importance (top 15)
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=X.columns[indices], palette="viridis")
plt.title("Top 15 Feature Importances for Startup Success")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

# 2. Save the trained model to the models/ directory
os.makedirs('../models', exist_ok=True)
model_path = '../models/success_model.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Success! XGBoost model saved successfully at:\n{model_path}")